In [2]:
from google.cloud import bigquery
from google.oauth2 import service_account

import psycopg2
import pandas as pd
from sqlalchemy import create_engine,URL

In [3]:
credentials = service_account.Credentials.from_service_account_file(
  'c:/Users/Bob/oasisbiz/datawarehouse-390004-34bcb00fb7cb.json'
)
project_id = 'datawarehouse-390004'

In [4]:
client = bigquery.Client(
  project=project_id,
  credentials=credentials
)

In [5]:
# connect to localhost
conn = psycopg2.connect(
  host='localhost',
  port=5432,
  database='postgres',
  user='postgres',
  password='postgres'
)
conn.set_session(autocommit=True)
cursor = conn.cursor()

In [35]:
engine = create_engine(
  URL.create(
    drivername='postgresql+psycopg2',
    host='localhost',
    port=5432,
    database='postgres',
    username='postgres',
    password='postgres'
  )
)

In [ ]:
# job_config = bigquery.LoadJobConfig(
#   schema=[
#     bigquery.SchemaField(name='poi_index',field_type='NUMERIC'),
#     bigquery.SchemaField(name='lat',field_type='NUMERIC'),
#     bigquery.SchemaField(name='lon',field_type='NUMERIC')
#   ],
#   create_disposition="CREATE_IF_NEEDED",
#   write_disposition="WRITE_APPEND",
#   source_format=bigquery.SourceFormat.CSV,
#   field_delimiter='|'
# )

In [ ]:
# job = client.query(
#   f'''
#   select
#     deal_amount,
#     contract_date,
#     floor_ floor,
#     plottage,
#     pnu,
#     st_x(trade_case_point) longitude,
#     st_y(trade_case_point) latitude,
#     building_type,
#     building_detail_type,
#     building_use,
#     build_year,
#     building_area,
#     sig_cd,
#     emd_cd,
#     land_use
#   from m2.cremao_real_estate_trade_case
#   where
#     sido_cd = '11' and
#     building_type = '집합' and
#     contract_date >= '2020-01-01'
#   '''
# )
# deal_df = job.result().to_dataframe()

c:\Users\Bob\AppData\Local\Programs\Python\Python312\Lib\site-packages\google\cloud\bigquery\table.py:1957: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [ ]:
# # create columns from contract_date
# deal_df['contract_year'] = [
#   date.year
#   for date
#   in deal_df['contract_date']
# ]
# deal_df['contract_month'] = [
#   date.month
#   for date
#   in deal_df['contract_date']
# ]
# deal_df['contract_day'] = [
#   date.day
#   for date
#   in deal_df['contract_date']
# ]

In [ ]:
# deal_df['building_age'] = deal_df['contract_year'] - deal_df['build_year'].astype('float')

In [99]:
# deal_df['geom'] = 'POINT (' + deal_df['longitude'].astype('string') + ' ' + deal_df['latitude'].astype('string') + ')'

In [ ]:
# deal_df = deal_df[[
#   'deal_amount','contract_year','contract_month','contract_day','floor','plottage','land_use','build_year','building_age','building_type','building_detail_type','building_use','building_area','sig_cd','emd_cd','pnu','longitude','latitude','geom'
# ]]

In [ ]:
# deal_df_clean = deal_df[0:0]

# for sig in deal_df['sig_cd'].unique():
#   tmp_df = deal_df[deal_df['sig_cd'] == sig]
#   # IQR 계산
#   Q1 = tmp_df['deal_amount'].quantile(0.25)
#   Q3 = tmp_df['deal_amount'].quantile(0.75)
#   IQR = Q3 - Q1

#   # # 이상치 제거
#   deal_df_clean = pd.concat([
#     deal_df_clean,
#     tmp_df[~((tmp_df['deal_amount'] < (Q1 - 1.5 * IQR)) | (tmp_df['deal_amount'] > (Q3 + 1.5 * IQR)))]
#   ])

In [ ]:
# print(len(deal_df_clean), '/', len(deal_df))

11668 / 12664


In [ ]:
# try:
#   cursor.execute(
#     f'''
#     create table m1_y (
#       deal_amount numeric,
#       contract_year numeric,
#       contract_month numeric,
#       contract_day numeric,
#       floor numeric,
#       plottage numeric,
#       land_use varchar,
#       build_year numeric,
#       building_age numeric,
#       building_type varchar,
#       building_detail_type varchar,
#       building_use varchar,
#       building_area numeric,
#       sig_cd varchar,
#       emd_cd varchar,
#       pnu varchar,
#       longitude varchar,
#       latitude varchar,
#       geom geometry(geometry,4326)
#     )
#     '''
#   )
# except Exception as err:
#   print(err)

오류:  "m1_y" 이름의 릴레이션(relation)이 이미 있습니다



In [ ]:
# engine = create_engine(
#   URL.create(
#     drivername='postgresql+psycopg2',
#     host='localhost',
#     port=5432,
#     database='postgres',
#     username='postgres',
#     password='postgres'
#   )
# )

In [ ]:
# try:
#   cursor.execute(
#     'delete from m1_y'
#   )
#   deal_df_clean.to_sql(
#     'm1_y',
#     engine,
#     if_exists='append',
#     index=False,
#   )
# except Exception as err:
#   print(err)

In [ ]:
# try:
#   cursor.execute(
#     f'''
#     alter table m1_y add geom_3857 geometry(geometry,3857);
#     update m1_y set geom_3857 = st_transform(geom,3857)
#     '''
#   )
# except Exception as err:
#   print(err)

In [ ]:
# try:
#   cursor.execute(
#     f'''
#     create index idx_m1_y_geom on m1_y using gist(geom);
#     create index idx_m1_y_geom_3857 on m1_y using gist(geom_3857)
#     '''
#   )
# except Exception as err:
#   print(err)

In [8]:
# read m1_y
cursor.execute(
  'select * from m1_y'
)
y_df = pd.DataFrame(
  cursor.fetchall(),
  columns=[col.name for col in cursor.description]
)

---
x 생성

In [9]:
# 건축물 정보 bld_info
cursor.execute(
f'''
select
	pnu,
	sum(plat_area) plat_area,
	sum(arch_area) arch_area,
	sum(tot_area) tot_area,
	sum(elev_cnt) elev_cnt,
	sum(parklot_cnt) parklot_cnt
from building_info
group by 1
'''
)
bld_info = pd.DataFrame(
  cursor.fetchall(),
  columns=[col.name for col in cursor.description]
)

In [10]:
# 시군구 매출 sig_sales
cursor.execute(
  f'''
  select
    extract('year' from base_dt) base_year,
    sig_cd,
    sum(store_cnt) sig_store_cnt,
    sum(sales_amt) sig_sales_amt,
    sum(sales_cnt) sig_sales_cnt,
    sum(delivery_amt) sig_delivery_amt,
    sum(delivery_cnt) sig_delivery_cnt,
    sum(sales_restaurant_amt) sig_sales_restaurant_amt,
    sum(sales_service_amt) sig_sales_service_amt,
    sum(sales_retail_amt) sig_sales_retail_amt,
    sum(sales_etc_amt) sig_sales_etc_amt,
    sum(sales_wk_amt) sig_sales_wk_amt,
    sum(sales_we_amt) sig_sales_we_amt,
    sum(sales_morning_amt) sig_sales_morning_amt,
    sum(sales_afternoon_amt) sig_sales_afternoon_amt,
    sum(sales_evening_amt) sig_sales_evening_amt,
    sum(sales_night_amt) sig_sales_night_amt,
    sum(sales_male_amt) sig_sales_male_amt,
    sum(sales_female_amt) sig_sales_female_amt,
    sum(sales_age20_amt) sig_sales_age20_amt,
    sum(sales_age30_amt) sig_sales_age30_amt,
    sum(sales_age40_amt) sig_sales_age40_amt,
    sum(sales_age50_amt) sig_sales_age50_amt,
    sum(sales_age60_amt) sig_sales_age60_amt
  from sig_sales
  group by 1,2
  '''
)
sig_sales = pd.DataFrame(
  cursor.fetchall(),
  columns=[col.name for col in cursor.description]
)

In [11]:
# 법정동 매출 emd_sales
cursor.execute(
  f'''
  select
    extract('year' from base_dt) base_year,
    emd_cd,
    sum(store_cnt) emd_store_cnt,
    sum(sales_amt) emd_sales_amt,
    sum(sales_cnt) emd_sales_cnt,
    sum(delivery_amt) emd_delivery_amt,
    sum(delivery_cnt) emd_delivery_cnt,
    sum(sales_restaurant_amt) emd_sales_restaurant_amt,
    sum(sales_service_amt) emd_sales_service_amt,
    sum(sales_retail_amt) emd_sales_retail_amt,
    sum(sales_etc_amt) emd_sales_etc_amt,
    sum(sales_wk_amt) emd_sales_wk_amt,
    sum(sales_we_amt) emd_sales_we_amt,
    sum(sales_morning_amt) emd_sales_morning_amt,
    sum(sales_afternoon_amt) emd_sales_afternoon_amt,
    sum(sales_evening_amt) emd_sales_evening_amt,
    sum(sales_night_amt) emd_sales_night_amt,
    sum(sales_male_amt) emd_sales_male_amt,
    sum(sales_female_amt) emd_sales_female_amt,
    sum(sales_age20_amt) emd_sales_age20_amt,
    sum(sales_age30_amt) emd_sales_age30_amt,
    sum(sales_age40_amt) emd_sales_age40_amt,
    sum(sales_age50_amt) emd_sales_age50_amt,
    sum(sales_age60_amt) emd_sales_age60_amt
  from emd_sales
  group by 1,2
  '''
)
emd_sales = pd.DataFrame(
  cursor.fetchall(),
  columns=[col.name for col in cursor.description]
)

In [ ]:
# 공시지가 lot_plp
cursor.execute(
  f'''
  select
    distinct on (pnu)
    base_year,
    pnu,
    amount plp_amt
  from public_land_price
  order by pnu,base_year desc
  '''
)
lot_plp = pd.DataFrame(
  cursor.fetchall(),
  columns=[col.name for col in cursor.description]
)
del lot_plp['base_year']

In [13]:
# 지하철 접근성 - 최단거리 lot_subway_dist
cursor.execute(
  f'''
  select
    y.pnu,
    round(st_distance(
      ST_Transform(y.geom,3857),
      ST_Transform(subway_ent.geom,3857)
    )) subway_dist
  from (
    select
      distinct on (pnu)
      pnu,
      geom
    from m1_y
  ) y,
  (
    select st_collect(geom) geom
    from subway_ent
  ) subway_ent
  '''
)
lot_subway_dist = pd.DataFrame(
  cursor.fetchall(),
  columns=[col.name for col in cursor.description]
)

In [14]:
# 지하철 접근성 - 반경 500m 내 역 개수 lot_subway_500_cnt
cursor.execute(
f'''select
	y.pnu,
	count(y.pnu) subway_500_cnt
from (
	select
		distinct on (pnu)
		pnu,
		geom
	from m1_y
) y,
(
	select
		station_nm,
		st_collect(geom) geom
	from subway_ent
	group by 1
) subway_ent
where
	st_dwithin(
		ST_Transform(y.geom,3857),
		ST_Transform(subway_ent.geom,3857),
		500
	)
group by 1'''
)
lot_subway_500_cnt = pd.DataFrame(
  cursor.fetchall(),
  columns=[col.name for col in cursor.description]
)

In [15]:
# 지하철 접근성 - 반경 1000m 내 역 개수 lot_subway_1000_cnt
cursor.execute(
f'''select
	y.pnu,
	count(y.pnu) subway_1000_cnt
from (
	select
		distinct on (pnu)
		pnu,
		geom
	from m1_y
) y,
(
	select
		station_nm,
		st_collect(geom) geom
	from subway_ent
	group by 1
) subway_ent
where
	st_dwithin(
		ST_Transform(y.geom,3857),
		ST_Transform(subway_ent.geom,3857),
		1000
	)
group by 1'''
)
lot_subway_1000_cnt = pd.DataFrame(
  cursor.fetchall(),
  columns=[col.name for col in cursor.description]
)

In [7]:
# 유동인구 - 반경 500m 내 lot_walk_pop
cursor.execute(
  f'''
  select
    extract('year' from base_dt) base_year,
    y.pnu,
    sum(tot_cnt) walk_tot_cnt,
    sum(male_cnt) walk_male_cnt,
    sum(female_cnt) walk_female_cnt,
    sum(age_u20_cnt) walk_age_u20_cnt,
    sum(age_20_cnt) walk_age_20_cnt,
    sum(age_30_cnt) walk_age_30_cnt,
    sum(age_40_cnt) walk_age_40_cnt,
    sum(age_50_cnt) walk_age_50_cnt,
    sum(age_o50_cnt) walk_age_o50_cnt,
    sum(time_0810_cnt) walk_morning_cnt,
    sum(time_1113_cnt + time_1416_cnt) walk_afternoon_cnt,
    sum(time_1719_cnt + time_2022_cnt) walk_evening_cnt,
    sum(time_2307_cnt) walk_time_night_cnt
  from (
    select
      distinct on (pnu)
      pnu,
      geom_3857
    from m1_y
  ) y,
  walk_pop
  where
    st_dwithin(
      y.geom_3857,
      walk_pop.geom_3857,
      500
    )
  group by 1,2
  '''
)
lot_walk_pop = pd.DataFrame(
  cursor.fetchall(),
  columns=[col.name for col in cursor.description]
)

In [22]:
lot_plp['base_year'].value_counts()

base_year
2024    893592
2023      6539
Name: count, dtype: int64

---
m1_y + x 통합

In [33]:
m1_total_df = y_df.merge(
  bld_info,
  how='left',
  on='pnu'
).merge(
  sig_sales,
  how='left',
  left_on=['sig_cd','contract_year'],
  right_on=['sig_cd','base_year']
).merge(
  emd_sales,
  how='left',
  left_on=['emd_cd','contract_year'],
  right_on=['emd_cd','base_year']
).merge(
  lot_plp,
  how='left',
  on='pnu'
).merge(
  lot_subway_dist,
  how='left',
  on='pnu'
).merge(
  lot_subway_500_cnt,
  how='left',
  on='pnu'
).merge(
  lot_subway_1000_cnt,
  how='left',
  on='pnu'
).merge(
  lot_walk_pop,
  how='left',
  on='pnu'
)

In [36]:
m1_total_df.to_sql(
  'm1_total',
  engine,
  if_exists='replace',
  index=False
)

354

---
전달용 데이터 저장하기

In [37]:
m1_total_df[
  ['deal_amount','contract_year','contract_month','contract_day','floor','plottage','land_use','build_year','building_age','building_type','building_detail_type','building_use','building_area','sig_cd','emd_cd','pnu','longitude','latitude','plat_area','arch_area','tot_area','elev_cnt','parklot_cnt','base_year_x','sig_store_cnt','sig_sales_amt','sig_sales_cnt','sig_delivery_amt','sig_delivery_cnt','sig_sales_restaurant_amt','sig_sales_service_amt','sig_sales_retail_amt','sig_sales_etc_amt','sig_sales_wk_amt','sig_sales_we_amt','sig_sales_morning_amt','sig_sales_afternoon_amt','sig_sales_evening_amt','sig_sales_night_amt','sig_sales_male_amt','sig_sales_female_amt','sig_sales_age20_amt','sig_sales_age30_amt','sig_sales_age40_amt','sig_sales_age50_amt','sig_sales_age60_amt','base_year_y','emd_store_cnt','emd_sales_amt','emd_sales_cnt','emd_delivery_amt','emd_delivery_cnt','emd_sales_restaurant_amt','emd_sales_service_amt','emd_sales_retail_amt','emd_sales_etc_amt','emd_sales_wk_amt','emd_sales_we_amt','emd_sales_morning_amt','emd_sales_afternoon_amt','emd_sales_evening_amt','emd_sales_night_amt','emd_sales_male_amt','emd_sales_female_amt','emd_sales_age20_amt','emd_sales_age30_amt','emd_sales_age40_amt','emd_sales_age50_amt','emd_sales_age60_amt','plp_amt','subway_dist','subway_500_cnt','subway_1000_cnt','base_year','walk_tot_cnt','walk_male_cnt','walk_female_cnt','walk_age_u20_cnt','walk_age_20_cnt','walk_age_30_cnt','walk_age_40_cnt','walk_age_50_cnt','walk_age_o50_cnt','walk_morning_cnt','walk_afternoon_cnt','walk_evening_cnt','walk_time_night_cnt']
].to_csv(
  'm1_data_06.18.csv',
  sep=',',
  index=False
)

Bigquery 업로드

In [95]:
# # load table from file
# poi_df.to_csv(
#   'poi_df.csv',
#   header=False,
#   index=False,
#   sep='|'
# )
# load_job = client.load_table_from_file(
#   open('poi_df.csv','rb'),
#   'temp.poi_df',
#   job_config=job_config
# )
# load_job.result()